# 02 — NMF local dentro dos temas selecionados (modelo final)

Segundo estágio: para cada tema global escolhido, um NMF próprio (Gensim, contagens BoW de
lemas spaCy) separa subtemas. K varia de 2 a 6 (`final_sentence_pipeline.local.k_range`);
escolhe-se o K de maior NPMI na fronteira NPMI × diversidade, o que na prática é o K de
maior NPMI. C_v é só reportado.

- **Pré-requisito:** `global_run` e `selected_topics` preenchidos no `params.yaml`
  (última seção de `01_nmf_global.ipynb`).
- **Saída:** `<run global>/restricted_gensim/topicNN/` com `grid.csv`, `topics.csv`,
  `nmf_results.csv`, `model.gensim` e `manifest.json`. Temas já concluídos são pulados.
- **Depois:** a seção final monta a amostra cega para revisão humana em `04-requirements`.

Código do modelo: `03-topic-modeling/scripts/run_restricted_nmf_gensim.py`.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "03-topic-modeling").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "03-topic-modeling" / "scripts"))
sys.path.insert(0, str(ROOT / "03-topic-modeling" / "notebooks"))
pd.set_option("display.max_colwidth", 200)

PARAMS_PATH = ROOT / "03-topic-modeling" / "configs" / "params.yaml"
params = yaml.safe_load(PARAMS_PATH.read_text(encoding="utf-8"))
FINAL = params["final_sentence_pipeline"]
SEED = params["seed"]

import run_restricted_nmf_gensim as nmf_local

RUN = nmf_local.pinned_global_run(params)
TEMAS = FINAL["selected_topics"]
assert TEMAS, "preencha final_sentence_pipeline.selected_topics no params.yaml"
LOCAL = RUN / FINAL["local_subdir"]
print("Run global:", RUN.name, "| temas:", TEMAS)

## 1. Treinar os NMFs locais

In [ ]:
nmf_local.run(RUN, TEMAS, SEED)

## 2. K escolhido por tema

In [ ]:
linhas = []
for t in TEMAS:
    pasta = LOCAL / f"topic{t:02d}"
    man = json.loads((pasta / "manifest.json").read_text(encoding="utf-8"))
    grid = pd.read_csv(pasta / "grid.csv").set_index("k")
    k = man["selected_k"]
    linhas.append({"tema": t, "frases": man["n_sentences"], "vocabulario": man["vocabulary_size"],
                   "sem_bow": man["unassigned"], "K": k,
                   "npmi": grid.loc[k, "npmi"], "diversidade": grid.loc[k, "diversity"],
                   "cv": grid.loc[k, "cv"]})
resumo = pd.DataFrame(linhas)
resumo.round(3)

In [ ]:
ncol = (len(TEMAS) + 1) // 2
fig, axes = plt.subplots(2, ncol, figsize=(3 * ncol, 5.5), sharex=True, squeeze=False)
for ax, t in zip(axes.flat, TEMAS):
    grid = pd.read_csv(LOCAL / f"topic{t:02d}" / "grid.csv")
    ax.plot(grid.k, grid.npmi, marker="o")
    ax2 = ax.twinx()
    ax2.plot(grid.k, grid.diversity, marker="s", color="tab:orange")
    k = int(resumo.set_index("tema").loc[t, "K"])
    ax.axvline(k, color="grey", ls="--", lw=1)
    ax.set_title(f"tema {t} (K={k})", fontsize=9)
for ax in list(axes.flat)[len(TEMAS):]:
    ax.axis("off")
fig.suptitle("NPMI (azul) e diversidade (laranja) por K; tracejado = K escolhido", fontsize=10)
fig.tight_layout()

## 3. Subtemas: termos e frases exemplares

Para cada subtema, as três frases de maior peso local, de vídeos distintos. Subtema `-1` =
frases sem palavras no dicionário local.

In [ ]:
for t in TEMAS:
    pasta = LOCAL / f"topic{t:02d}"
    sub = pd.read_csv(pasta / "topics.csv")
    res = pd.read_csv(pasta / "nmf_results.csv", dtype={"sent_id": str, "post_id": str})
    res["peso_local"] = [max(json.loads(d)) if json.loads(d) else 0.0
                         for d in res.local_topic_distribution]
    print(f"\n################ Tema global {t} ({len(res)} frases)")
    for _, s in sub.iterrows():
        print(f"\n  -- {t}.{s.topic_id} ({s.n_sentences} frases): {s.top_terms}")
        top = (res.loc[res.local_topic_id == s.topic_id]
                  .sort_values("peso_local", ascending=False)
                  .drop_duplicates("post_id").head(3))
        for _, r in top.iterrows():
            print(f"     [{r.peso_local:.2f}|{r.product_category}] {r.sentence[:150]}")

## 4. Montar a amostra cega para revisão humana

Junta NMF global e local por `sent_id`, marca o léxico de pedido/queixa, gera os rankings
(tema, léxico, combinação, aleatório) e a amostra cega em
`04-requirements/data/output/topicos_requisitos/`. Nenhuma frase é rotulada como requisito:
`requirement_candidate` fica vazio para a anotação humana (guia em
`04-requirements/guia_anotacao_rq4.md`).

Antes de anotar, veja a idealização §5.3C (empate do ranking por tema) e §4.7A (protocolo de
teste).

In [ ]:
sys.path.insert(0, str(ROOT / "04-requirements"))
from nmf_requirement_review import run_nmf_review

SAIDA = (ROOT / "04-requirements" / "data" / "output" / "topicos_requisitos"
         / (FINAL.get("review_output") or f"{RUN.name}_revisao"))
resumo_revisao = run_nmf_review(
    RUN, TEMAS, SAIDA, local_subdir=FINAL["local_subdir"],
    expected_count=FINAL.get("expected_sentences"),
    top_k=FINAL["top_k"], n_random=FINAL["n_random"], seed=SEED,
)
print("Saída:", SAIDA)
{k: v for k, v in resumo_revisao.items() if k not in ("sources_sha256", "lexicon")}